### CREATING A MODEL
The API is designed for the management and creation of n-dimensional parametric models. It is still under development and currently does not support creating a model directly from a function. Defining a model is very simple; just create a class instance of `FittableModel` that has an `evaluate` method.


In [1]:
#import FittableModel
from base import FittableModel
import numpy as np

Now, let's define a generic class, for example, a 2D Gaussian. The API reads the first entries of the `evaluate` function and tries to determine the grid on which the model is defined by looking for the pattern `x`, `y`, and `z` for models from 1 to 3 dimensions.


In [2]:
class GenericGaussian2D(FittableModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def evaluate(self, x, y, amp, x0, y0, sigma_x, sigma_y, theta):
        """Two dimensional Gaussian function."""
        cost2 = np.cos(theta) ** 2
        sint2 = np.sin(theta) ** 2
        sin2t = np.sin(2.0 * theta)
        xstd2 = sigma_x**2
        ystd2 = sigma_y**2
        xdiff = x - x0
        ydiff = y - y0
        a = 0.5 * ((cost2 / xstd2) + (sint2 / ystd2))
        b = 0.5 * ((sin2t / xstd2) - (sin2t / ystd2))
        c = 0.5 * ((sint2 / xstd2) + (cost2 / ystd2))
        return amp * np.exp(-((a * xdiff**2) + (b * xdiff * ydiff) + (c * ydiff**2)))

We can now print the model on the screen. The API will create a user-friendly representation of our model to better understand it.

We can see that, unless specified otherwise, all parameter values are set to 1 with no restrictions on bounds. Additionally, since we did not specify anything, the API tried to determine on which parameters the grid is defined, correctly identifying that they are `x` and `y`.



In [3]:
my_gaussian = GenericGaussian2D()

print(my_gaussian)

MODEL NAME: GenericGaussian2D 
FREE PARAMS: 6
GRID VARIABLES: ['x', 'y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             1.00       0          (-inf, inf)         
1    x0              1.00       0          (-inf, inf)         
2    y0              1.00       0          (-inf, inf)         
3    sigma_x         1.00       0          (-inf, inf)         
4    sigma_y         1.00       0          (-inf, inf)         
5    theta           1.00       0          (-inf, inf)         



#### SPECIFY A GRID FOR THE MODEL

Attention!! When defining your own model, it is important to understand what happens under the hood to avoid unexpected behavior in the future. When you create an instance of `FittableModel`, the API always looks for a grid to define the number of dimensions, inputs, and outputs of the model you want to create! These entries can be overwritten at any time by the keys *'N_INPUTS'*, *'N_DIMS'*, and *'N_OUTPUTS'* defined as class attributes.

- __*'N_INPUTS'*__ defines the number of inputs the model expects besides the parameters, i.e., the grid on which it is defined. A number of inputs equal to 2 indicates that the first 2 entries of the model are the grid.
- Similarly, __*'N_DIMS'*__ indicates the number of dimensions of the model.
- __*'N_OUTPUTS'*__ indicates how many outputs the model has. A value of 1 means the model has a single output (in this case, the output will be the intensity on the z-axis).


In [4]:
class GenericGaussian2D_new(FittableModel):
    N_INPUTS = 1
    N_OUTPUTS = 0
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def evaluate(self, x, y, amp, x0, y0, sigma_x, sigma_y, theta):
        """Two dimensional Gaussian function."""
        cost2 = np.cos(theta) ** 2
        sint2 = np.sin(theta) ** 2
        sin2t = np.sin(2.0 * theta)
        xstd2 = sigma_x**2
        ystd2 = sigma_y**2
        xdiff = x - x0
        ydiff = y - y0
        a = 0.5 * ((cost2 / xstd2) + (sint2 / ystd2))
        b = 0.5 * ((sin2t / xstd2) - (sin2t / ystd2))
        c = 0.5 * ((sint2 / xstd2) + (cost2 / ystd2))
        return amp * np.exp(-((a * xdiff**2) + (b * xdiff * ydiff) + (c * ydiff**2)))

my_gaussian2 = GenericGaussian2D_new()


print(my_gaussian2)

MODEL NAME: GenericGaussian2D_new 
FREE PARAMS: 7
GRID VARIABLES: ['x']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    y               1.00       0          (-inf, inf)         
1    amp             1.00       0          (-inf, inf)         
2    x0              1.00       0          (-inf, inf)         
3    y0              1.00       0          (-inf, inf)         
4    sigma_x         1.00       0          (-inf, inf)         
5    sigma_y         1.00       0          (-inf, inf)         
6    theta           1.00       0          (-inf, inf)         



We can see that, by changing *'N_INPUTS'* to 1, the key `y` in the `evaluate` method is no longer considered as part of the grid but as a parameter of the model.



This behavior is exactly what an advanced user might desire if they want to define the grid keys with names different from `x`/`y`, `z`, or similar. To make everything more intuitive, the API provides classes that define standard models in n-dimensions, taking the first n-th entries of the `evaluate` method and transforming them into a grid for the model.

**ATTENTION!** The grid on which the model is defined must necessarily represent the first entry of the `evaluate` method. I am currently working to implement a more flexible grid management system, but for now, these are the rules.



The default classes provided are: `Fittable1D`, `Fittable2D`, `Kernel1D`, and `Kernel2D`.


| Class Name    | N_DIMENSIONS | IS_COMPOSITE | N_INPUTS | N_OUTPUTS |
|---------------|--------------|--------------|----------|-----------|
| Fittable1D    | 1            | False        | 1        | 1         |
| Fittable2D    | 2            | False        | 2        | 1         |
| Kernel2D      | 2            | False        | 1        | 1         |
| Kernel1D      | 1            | False        | 1        | 1         |


In [5]:
from zoo import Fittable2D


class Gaussian2D(Fittable2D):
    def evaluate(self, grid_x, grid_y, amp, x0, y0, sigma_x, sigma_y, theta):
        """Two dimensional Gaussian function."""
        cost2 = np.cos(theta) ** 2
        sint2 = np.sin(theta) ** 2
        sin2t = np.sin(2.0 * theta)
        xstd2 = sigma_x**2
        ystd2 = sigma_y**2
        xdiff = grid_x - x0
        ydiff = grid_y - y0
        a = 0.5 * ((cost2 / xstd2) + (sint2 / ystd2))
        b = 0.5 * ((sin2t / xstd2) - (sin2t / ystd2))
        c = 0.5 * ((sint2 / xstd2) + (cost2 / ystd2))
        return amp * np.exp(-((a * xdiff**2) + (b * xdiff * ydiff) + (c * ydiff**2)))


gaussian = Gaussian2D()


print(gaussian)

MODEL NAME: Gaussian2D 
FREE PARAMS: 6
GRID VARIABLES: ['grid_x', 'grid_y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             1.00       0          (-inf, inf)         
1    x0              1.00       0          (-inf, inf)         
2    y0              1.00       0          (-inf, inf)         
3    sigma_x         1.00       0          (-inf, inf)         
4    sigma_y         1.00       0          (-inf, inf)         
5    theta           1.00       0          (-inf, inf)         



We can see that using `Fittable2D` also saves us from initializing the parent class. The grid behavior is as we expect.


#### INIZIALISING PARAMETERS

There are several ways to initialize the parameters of a model. This is not the recommended method, as parameters and their attributes can be changed at any time. However, if preferred (who am I to judge), models can be initialized with preset values and/or bounds.


The fastest method is to directly use the parameter names with their respective values. It is quick but only allows managing individual values.


In [6]:
gaussian_with_params = Gaussian2D(
    name = 'My Gaussian With Parameters',
    amp = 1.2,
    x0 = 1,
    y0 = 1.3,
    sigma_x = 4,
)

print(gaussian_with_params)

MODEL NAME: My Gaussian With Parameters 
FREE PARAMS: 6
GRID VARIABLES: ['grid_x', 'grid_y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             1.20       0          (-inf, inf)         
1    x0              1.00       0          (-inf, inf)         
2    y0              1.30       0          (-inf, inf)         
3    sigma_x         4.00       0          (-inf, inf)         
4    sigma_y         1.00       0          (-inf, inf)         
5    theta           1.00       0          (-inf, inf)         



Alternatively, you can provide a list of `Parameters` using the key `parameters=[...]`. This allows access to attributes such as bounds, descriptions, and whether the parameter is a constant (frozen).

**ATTENTION!** In both cases, trying to access a parameter name that is not present in the `evaluate` method of the original class will result in an error! You can try this.


In [7]:
from parameters import Parameter

another_gaussian = Gaussian2D(
    parameters =[
        Parameter('amp', value=22, frozen=True, bounds=(0,55)),
        Parameter('theta', value =np.pi/2, bounds = (-np.pi, np.pi))
    ],
    name = 'Another gaussian'
)

print(another_gaussian)

MODEL NAME: Another gaussian 
FREE PARAMS: 5
GRID VARIABLES: ['grid_x', 'grid_y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             22.00      1          (0.00, 55.00)       
1    x0              1.00       0          (-inf, inf)         
2    y0              1.00       0          (-inf, inf)         
3    sigma_x         1.00       0          (-inf, inf)         
4    sigma_y         1.00       0          (-inf, inf)         
5    theta           1.57       0          (-3.14, 3.14)       



#### ACCESSING TO PARAMETERS

Accessing parameters is very simple. Although `FittableModel` implements methods that allow accessing multiple models together, accessing a parameter in a model is the same as accessing a key in a dictionary.


In [8]:
print(another_gaussian["amp"])

# we can also modify the parameters, this will affect the model itself
another_gaussian["amp"].frozen = False
another_gaussian["amp"].value = 1.67
another_gaussian["amp"].bounds = (-2,4.44)

another_gaussian['sigma_x'].frozen = True

print(another_gaussian)

PARAM NAME: amp
____________________________________________________________________________________________________
NOME            VALORE     FREEZE     BOUNDS               DESCRIZIONE          
____________________________________________________________________________________________________
amp             22         1          (0, 55)                                   

MODEL NAME: Another gaussian 
FREE PARAMS: 5
GRID VARIABLES: ['grid_x', 'grid_y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             1.67       0          (-2.00, 4.44)       
1    x0              1.00       0          (-inf, inf)         
2    y0              1.00       0          (-inf, inf)         
3    sigma_x         1.00       1          (-inf, inf)         
4    sigma_y         1.00       0          (-inf, inf)         
5    theta 

#### FREEZING PARAMETERS

The API supports freezing parameters. The value of a frozen parameter cannot be changed, otherwise a fatal error will occur. Additionally, any keys in the `evaluate` method that are already set to a fixed value are interpreted as constant parameters and frozen by default (don't worry, they can always be unfrozen later).


In [9]:
'''try:
    another_gaussian["sigma_x"].value = 2
except KeyError as e:
    print(e)'''

'try:\n    another_gaussian["sigma_x"].value = 2\nexcept KeyError as e:\n    print(e)'

We can see that if, for example, the key `theta` is already set to a value inside the `evaluate` method, it will be interpreted as a frozen parameter.

A frozen parameter will affect the number of free parameters!

In [10]:
class Gaussian2D_frozen(Fittable2D):
    def evaluate(self, grid_x, grid_y, amp, x0, y0, sigma_x, sigma_y, theta=np.pi):
        """Two dimensional Gaussian function."""
        cost2 = np.cos(theta) ** 2
        sint2 = np.sin(theta) ** 2
        sin2t = np.sin(2.0 * theta)
        xstd2 = sigma_x**2
        ystd2 = sigma_y**2
        xdiff = grid_x - x0
        ydiff = grid_y - y0
        a = 0.5 * ((cost2 / xstd2) + (sint2 / ystd2))
        b = 0.5 * ((sin2t / xstd2) - (sin2t / ystd2))
        c = 0.5 * ((sint2 / xstd2) + (cost2 / ystd2))
        return amp * np.exp(-((a * xdiff**2) + (b * xdiff * ydiff) + (c * ydiff**2)))


gaussian_f = Gaussian2D_frozen()


print(gaussian_f)

# NOTE we can alway unfreeze the param:
# gaussian_f['theta'].froen = False

MODEL NAME: Gaussian2D_frozen 
FREE PARAMS: 5
GRID VARIABLES: ['grid_x', 'grid_y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             1.00       0          (-inf, inf)         
1    x0              1.00       0          (-inf, inf)         
2    y0              1.00       0          (-inf, inf)         
3    sigma_x         1.00       0          (-inf, inf)         
4    sigma_y         1.00       0          (-inf, inf)         
5    theta           3.14       1          (-inf, inf)         



#### ACCESS TO MULTPLE PARAMETERS

We can access multiple parameters at once by using the standard methods of `FittableModel`: `set_parameters_values`, `set_parameters_bounds`, `freeze_parameters`, `unfreeze_parameters`.

accessing to multiple parameters can be done via keywords and dictionaries for all methods

In [11]:
#NOTE freeze parameters accepts bot the name of the param and the index of the same
# inside the model!
gaussian_f.freeze_parameters('amp', 'sigma_x')# equivalent to freeze_parameters(0,3)

# 
gaussian_f.set_parameter_bounds(amp = (-10,99), 
                                theta = (0,5))


print(gaussian_f)

MODEL NAME: Gaussian2D_frozen 
FREE PARAMS: 3
GRID VARIABLES: ['grid_x', 'grid_y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             1.00       1          (-10.00, 99.00)     
1    x0              1.00       0          (-inf, inf)         
2    y0              1.00       0          (-inf, inf)         
3    sigma_x         1.00       1          (-inf, inf)         
4    sigma_y         1.00       0          (-inf, inf)         
5    theta           3.14       1          (0.00, 5.00)        



When freezing a parameter, you can also provide a new value to bind it to:


In [12]:
# NOTE freeze parameters accepts bot the name of the param and the index of the same
# inside the model!
gaussian_f.freeze_parameters(y0 = 1.23)  # equivalent to freeze_parameters({'theta':1.23})

#
print(gaussian_f)

MODEL NAME: Gaussian2D_frozen 
FREE PARAMS: 2
GRID VARIABLES: ['grid_x', 'grid_y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             1.00       1          (-10.00, 99.00)     
1    x0              1.00       0          (-inf, inf)         
2    y0              1.23       1          (-inf, inf)         
3    sigma_x         1.00       1          (-inf, inf)         
4    sigma_y         1.00       0          (-inf, inf)         
5    theta           3.14       1          (0.00, 5.00)        



Or we can freeze all parameters at once

In [13]:
gaussian_f.freeze_parameters()
print(gaussian_f)

MODEL NAME: Gaussian2D_frozen 
FREE PARAMS: 0
GRID VARIABLES: ['grid_x', 'grid_y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             1.00       1          (-10.00, 99.00)     
1    x0              1.00       1          (-inf, inf)         
2    y0              1.23       1          (-inf, inf)         
3    sigma_x         1.00       1          (-inf, inf)         
4    sigma_y         1.00       1          (-inf, inf)         
5    theta           3.14       1          (0.00, 5.00)        



... and unfreeze all by following the sam logic

In [14]:
gaussian_f.unfreeze_parameters()  # all

print(gaussian_f)


MODEL NAME: Gaussian2D_frozen 
FREE PARAMS: 6
GRID VARIABLES: ['grid_x', 'grid_y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             1.00       0          (-10.00, 99.00)     
1    x0              1.00       0          (-inf, inf)         
2    y0              1.23       0          (-inf, inf)         
3    sigma_x         1.00       0          (-inf, inf)         
4    sigma_y         1.00       0          (-inf, inf)         
5    theta           3.14       0          (0.00, 5.00)        



Now we can set new parameters values:

Alwasy remember the values must be containied inside the bounds!

In [15]:
gaussian_f.set_parameters_values(amp=11, sigma_x=11, x0=2, y0=3.4)

# equivalent to gaussian_f.set_parameters_values({'amp':11, 'sigma_x':11, 'x0':2, 'y0':3.4})
print(gaussian_f)

MODEL NAME: Gaussian2D_frozen 
FREE PARAMS: 6
GRID VARIABLES: ['grid_x', 'grid_y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             11.00      0          (-10.00, 99.00)     
1    x0              2.00       0          (-inf, inf)         
2    y0              3.40       0          (-inf, inf)         
3    sigma_x         11.00      0          (-inf, inf)         
4    sigma_y         1.00       0          (-inf, inf)         
5    theta           3.14       0          (0.00, 5.00)        



#### ACCESSING TO OTHER ATTRIBUTES

Each model has a corresponding dictionary of parameters, which can be viewed through the `parameters` method.

Modifying the attributes of the parameter dictionary will change their values. It is highly recommended not to access the parameters through this method.


In [16]:
print(gaussian_f.parameters)

gaussian_f.parameters['amp'].value = 22

print(gaussian_f)

{'amp': <parameters.Parameter object at 0x7f5463a3f390>, 'x0': <parameters.Parameter object at 0x7f5463a3f410>, 'y0': <parameters.Parameter object at 0x7f5463a3f450>, 'sigma_x': <parameters.Parameter object at 0x7f5463a3f490>, 'sigma_y': <parameters.Parameter object at 0x7f5463a3f510>, 'theta': <parameters.Parameter object at 0x7f5463a3f5d0>}
MODEL NAME: Gaussian2D_frozen 
FREE PARAMS: 6
GRID VARIABLES: ['grid_x', 'grid_y']
______________________________________________________________________
     NOME            VALORE     FREEZE     BOUNDS              
______________________________________________________________________
0    amp             22.00      0          (-10.00, 99.00)     
1    x0              2.00       0          (-inf, inf)         
2    y0              3.40       0          (-inf, inf)         
3    sigma_x         11.00      0          (-inf, inf)         
4    sigma_y         1.00       0          (-inf, inf)         
5    theta           3.14       0          (0.

We can see the values, bounds, and other attributes of the parameters through the methods `parameters_values`, `parameters_bounds`, `parameters_names`, and others.


In [17]:
print("NAMES:", gaussian_f.parameters_names)
print("VALUES:", gaussian_f.parameters_values)
print("BOUNDS:", gaussian_f.parameters_bounds)

print("FREE PARAMETERS", gaussian_f.free_parameters)
print("FROZEN PARAMS:", gaussian_f.frozen_parameters)

print("FROZEN MASK:", gaussian_f._binary_freeze_map)
print("UNFROZEN MASK:", gaussian_f._binary_melt_map)

print("NUMBER OF PARAMS:", gaussian_f.n_parameters)
print("NUMBER OF FREE PARAMS:", gaussian_f.n_free_parameters)

NAMES: ['amp', 'x0', 'y0', 'sigma_x', 'sigma_y', 'theta']
VALUES: [22, 2, 3.4, 11, 1, 3.141592653589793]
BOUNDS: [(-10, 99), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (0, 5)]
FREE PARAMETERS [<parameters.Parameter object at 0x7f5463a3f390>, <parameters.Parameter object at 0x7f5463a3f410>, <parameters.Parameter object at 0x7f5463a3f450>, <parameters.Parameter object at 0x7f5463a3f490>, <parameters.Parameter object at 0x7f5463a3f510>, <parameters.Parameter object at 0x7f5463a3f5d0>]
FROZEN PARAMS: []
FROZEN MASK: [False, False, False, False, False, False]
UNFROZEN MASK: [True, True, True, True, True, True]
NUMBER OF PARAMS: 6
NUMBER OF FREE PARAMS: 6


#### ITERATING AND CONDITIONALS

we can check if a parameter name is contained inside the model:


In [18]:
print('amp' in gaussian_f)

print('not_amp' in gaussian_f)

True
False


we can also iterate over the model to see all parameters:

In [19]:
print(len(gaussian_f))

for i, param in enumerate(gaussian_f):
    print(i, param)

6
0 PARAM NAME: amp
____________________________________________________________________________________________________
NOME            VALORE     FREEZE     BOUNDS               DESCRIZIONE          
____________________________________________________________________________________________________
amp             22         0          (-10, 99)                                 

1 PARAM NAME: x0
____________________________________________________________________________________________________
NOME            VALORE     FREEZE     BOUNDS               DESCRIZIONE          
____________________________________________________________________________________________________
x0              2          0          (-inf, inf)                               

2 PARAM NAME: y0
____________________________________________________________________________________________________
NOME            VALORE     FREEZE     BOUNDS               DESCRIZIONE          
__________________________________